In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
import scipy.stats as stats

sns.set_style('whitegrid')
sns.set_palette('Set2')

# Load cleaned data
df = pd.read_csv('data/skinport_items_cleaned(1).csv')

# Filter to the 4 weapons used in analysis
weapon_order = ['AK-47', 'M4A1-S', 'M4A4', 'AWP']
df_weapons = df[df['weapon'].isin(weapon_order)].copy()
df_weapons['avg'] = pd.to_numeric(df_weapons['avg'], errors='coerce')
df_weapons = df_weapons.dropna(subset=['avg'])

# Log-transform skewed variables
df_weapons['log_avg']    = np.log(df_weapons['avg'])
df_weapons['log_volume'] = np.log(df_weapons['volume'])


---
## 3. Multiple Linear Regression (MLR) Model

We build a Multiple Linear Regression model to predict CS2 skin prices from their attributes. Because `avg` is highly right-skewed (as seen in the histograms above), we apply a **log transformation** to stabilize variance and satisfy MLR's homoscedasticity assumption. Similarly, `volume` is log-transformed since sales activity follows a multiplicative pattern.

The response variable is:
$$\log(\text{avg\_price}) = \beta_0 + \beta_1\,\text{condition} + \beta_2\,\text{souvenir} + \beta_3\,\mathbf{1}_{\text{AWP}} + \beta_4\,\mathbf{1}_{\text{M4A1-S}} + \beta_5\,\mathbf{1}_{\text{M4A4}} + \beta_6\,\log(\text{volume}) + \varepsilon$$

*(AK-47 is the reference weapon category)*

---
### 3.1 Model Comparison

We fit three nested models of increasing complexity and compare them using **Adjusted R²**, **AIC**, and **BIC**.  
Lower AIC/BIC indicates a better balance of fit vs. complexity. `stat_trak` was tested but dropped — it was not statistically significant (p = 0.57).

In [ ]:
m1 = smf.ols('log_avg ~ condition',                                          data=df_weapons).fit()
m2 = smf.ols('log_avg ~ condition + souvenir + stat_trak',                   data=df_weapons).fit()
m3 = smf.ols('log_avg ~ condition + souvenir + C(weapon) + log_volume',      data=df_weapons).fit()

rows = []
for name, m in [
    ('M1: condition only',                m1),
    ('M2: + souvenir + stat_trak',        m2),
    ('M3: + weapon + log_volume (final)', m3),
]:
    rows.append({
        'Model':    name,
        'k':        int(m.df_model) + 2,
        'R²':       round(m.rsquared, 4),
        'Adj. R²':  round(m.rsquared_adj, 4),
        'AIC':      round(m.aic, 1),
        'BIC':      round(m.bic, 1),
    })

pd.DataFrame(rows)


---
### 3.2 Final Model Summary

M3 achieves the lowest AIC and BIC and the highest Adjusted R². We adopt it as our final model.

In [ ]:
print(m3.summary())


---
### 3.3 Coefficient Interpretation

Because the response is log-transformed, coefficients are interpreted on a **multiplicative scale**: a coefficient $\hat{\beta}$ means the predicted price is multiplied by $e^{\hat{\beta}}$ for a one-unit increase in that predictor, holding all others constant.

| Predictor | Coef | $e^{\hat{\beta}}$ | Interpretation |
|---|---|---|---|
| Condition (+1 tier) | +0.661 | ×1.94 | Each wear tier roughly doubles the price |
| Souvenir | −0.393 | ×0.68 | Souvenir skins are ~32% cheaper on average |
| AWP vs AK-47 | −0.210 | ×0.81 | AWP skins slightly cheaper than AK-47 |
| M4A1-S vs AK-47 | −0.692 | ×0.50 | M4A1-S skins ~50% cheaper than AK-47 |
| M4A4 vs AK-47 | −1.372 | ×0.25 | M4A4 skins ~75% cheaper than AK-47 |
| log(volume) (+1%) | −0.633 | ×0.53 | Higher-volume (common) skins are cheaper |


---
### 3.4 Diagnostic Plots

We verify the four standard MLR assumptions:
1. **Residuals vs. Fitted** — linearity and homoscedasticity
2. **Normal Q-Q** — normality of residuals
3. **Scale-Location** — constant residual spread
4. **Residuals vs. Leverage** — influential observations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('MLR Diagnostic Plots — Final Model (M3)', fontsize=14, fontweight='bold')

fitted    = m3.fittedvalues
resid     = m3.resid
std_resid = resid / resid.std()

# 1. Residuals vs Fitted
ax = axes[0, 0]
ax.scatter(fitted, resid, alpha=0.3, s=15, color='steelblue')
ax.axhline(0, color='red', linewidth=1.2, linestyle='--')
ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals')
ax.set_title('Residuals vs. Fitted')

# 2. Q-Q Plot
ax = axes[0, 1]
(osm, osr), (slope, intercept, _) = stats.probplot(resid, dist='norm')
ax.scatter(osm, osr, alpha=0.3, s=15, color='steelblue')
ax.plot(osm, slope * np.array(osm) + intercept, color='red', linewidth=1.5)
ax.set_xlabel('Theoretical Quantiles'); ax.set_ylabel('Sample Quantiles')
ax.set_title('Normal Q-Q Plot')

# 3. Scale-Location
ax = axes[1, 0]
ax.scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.3, s=15, color='steelblue')
ax.set_xlabel('Fitted values'); ax.set_ylabel('√|Standardized Residuals|')
ax.set_title('Scale-Location')

# 4. Residuals vs Leverage
leverage = m3.get_influence().hat_matrix_diag
ax = axes[1, 1]
ax.scatter(leverage, std_resid, alpha=0.3, s=15, color='steelblue')
ax.axhline(0, color='red', linewidth=1.2, linestyle='--')
ax.set_xlabel('Leverage'); ax.set_ylabel('Standardized Residuals')
ax.set_title('Residuals vs. Leverage')

plt.tight_layout()
plt.show()


---
### 3.5 Model Limitations

- **R² = 0.31** — condition, weapon type, and volume explain about 31% of log-price variance. The remaining variance likely reflects **skin rarity and visual desirability**, which are not available in the Skinport API.
- The Q-Q plot shows mild **heavy tails**, meaning residuals are slightly non-normal — a small number of ultra-rare skins (e.g., AK-47 Fire Serpent) produce large outliers the model cannot fully account for.
- **Souvenir** (p = 0.056) is borderline significant and may not hold in every subsample.
- The model covers only 4 weapons; generalization to other weapon classes would require additional data.
